In [1]:


# Cell 1: Setup path and imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src import CustomBPETokenizer, normalize_review
from datasets import load_dataset

# Cell 2+: Use classes directly
my_tokenizer = CustomBPETokenizer()
model_path = Path.cwd().parent / "data" / "models" / "tokenizer_state.json"


c:\Users\pc\anaconda3\envs\drishti\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_dataset = load_dataset("stanfordnlp/imdb")

In [3]:

train_val_split = raw_dataset["train"].train_test_split(test_size=0.2, seed=42)

datasets = {
    "train": train_val_split["train"],
    "validation": train_val_split["test"],
    "test": raw_dataset["test"]
}

print(f"Train size: {len(datasets['train'])}")
print(f"Validation size: {len(datasets['validation'])}")
print(f"Test size: {len(datasets['test'])}\n")



Train size: 20000
Validation size: 5000
Test size: 25000



In [4]:
cleaned_datasets = {}
for split_name, split_data in datasets.items():
    cleaned_datasets[split_name] = split_data.map(normalize_review, num_proc=4)

print("\n--- Normalization Complete ---")
cleaned_datasets


--- Normalization Complete ---


{'train': Dataset({
     features: ['text', 'label'],
     num_rows: 20000
 }),
 'validation': Dataset({
     features: ['text', 'label'],
     num_rows: 5000
 }),
 'test': Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 })}

In [5]:
corpus = []
corpus = [i for i in cleaned_datasets['train']['text']]


In [11]:

my_tokenizer.load_or_train(corpus=corpus, vocab_size=10000, file_path=str(model_path))

Tokenizer state loaded from c:\Users\pc\Desktop\LLMs\data\models\tokenizer_state.json. Vocabulary size: 10004


In [12]:

test_text = "This is a good Movie, I like Avengers, but I don't like the ending. The acting was great, but the plot was a bit weak."

input_ids = my_tokenizer.encode(test_text)
print("Input IDs:", input_ids)
print("Number of tokens:", len(input_ids))
reconstructed_text = my_tokenizer.decode(input_ids)
print("Reconstructed:", reconstructed_text)

['this', 'Ġis', 'Ġa', 'Ġgood', 'Ġmovie', ',', 'Ġi', 'Ġlike', 'Ġaven', 'gers', ',', 'Ġbut', 'Ġi', 'Ġdon', "'", 't', 'Ġlike', 'Ġthe', 'Ġending', '.', 'Ġthe', 'Ġacting', 'Ġwas', 'Ġgreat', ',', 'Ġbut', 'Ġthe', 'Ġplot', 'Ġwas', 'Ġa', 'Ġbit', 'Ġweak', '.']
Input IDs: [761, 117, 79, 298, 176, 3, 86, 245, 8088, 4332, 3, 177, 86, 405, 45, 5, 245, 83, 1012, 3, 83, 512, 154, 384, 3, 177, 83, 489, 154, 79, 721, 1749, 3]
Number of tokens: 33
Reconstructed: this is a good movie<|endoftext|> i like avengers<|endoftext|> but i don't like the ending<|endoftext|> the acting was great<|endoftext|> but the plot was a bit weak<|endoftext|>


In [15]:
from src.data import create_dataloaders

# Assuming you already have your lists of texts and labels from HuggingFace loaded 
# as `train_texts`, `train_labels`, etc., and your `tokenizer` instantiated.

train_loader, val_loader, test_loader = create_dataloaders(
    train_texts, train_labels, 
    val_texts, val_labels, 
    test_texts, test_labels, 
    tokenizer=tokenizer, 
    batch_size=16, 
    max_length=128
)

# Fetch exactly one batch to inspect
sample_batch_tokens, sample_batch_labels = next(iter(train_loader))

print("Token tensor shape:", sample_batch_tokens.shape)
print("Label tensor shape:", sample_batch_labels.shape)
print("First review token IDs:", sample_batch_tokens[0])
print("First review label:", sample_batch_labels[0])

ImportError: cannot import name 'create_dataloaders' from 'src.data' (c:\Users\pc\Desktop\LLMs\src\data.py)